# Aboveground Carbon Across Biomes

Compares aboveground carbon density (Mg C/ha) across four contrasting forest types using ESA CCI Biomass, restricted to forest-cover pixels identified via unsupervised clustering of AlphaEarth satellite embeddings (cross-referenced with Hansen tree cover to label the forest cluster). Also compares the four zones structurally using their forest-only AlphaEarth embedding signatures.

**Zones:** boreal managed forest (Abitibi, Quebec), intact tropical rainforest (Tapajos, Brazil), native temperate forest (Alerce Costero, Chile), and an even-aged Pinus radiata plantation (Biobio, Chile).

**Note on GEDI:** an earlier version of this notebook cross-checked ESA CCI against GEDI L4B (1 km gridded biomass). That comparison was dropped after diagnostics showed it was unreliable at this AOI scale: GEDI L4B's `MU` band is defined as the mean biomass 'including forest and non-forest' within each 1 km cell (not a forest-only value, so partial-cover cells can't be corrected after the fact by masking), and real GEDI sampling density varied sharply across zones (e.g. Tapajos averaged under 1 ground track per 1 km cell, meaning most of its 'data' was a statistical fill-in rather than a direct measurement). See the README for the diagnostic numbers.

**Pipeline:** define zones -> AlphaEarth clustering + forest-mask identification -> carbon from ESA CCI Biomass (forest-weighted) -> comparison chart -> forest-only AlphaEarth signatures -> similarity heatmap.

In [ ]:
import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')

from zones import ZONES, get_zone_geometry, get_zone_label
from carbon_sources import get_esa_cci_carbon, forest_weighted_mean_carbon
from embedding_utils import cosine_similarity_matrix, cluster_and_identify_forest

PROJECT = "your-gee-project-id"
ee.Initialize(project=PROJECT)

## 1. Study zones

Small bounding boxes (~20-25 km) — illustrative placeholders centered on well-known sites for each forest type. Adjust in `src/zones.py` if you have more precise boundaries.

In [ ]:
for key in ZONES:
    geom = get_zone_geometry(key)
    area_ha = geom.area().divide(10000).getInfo()
    print(f"{key}: {get_zone_label(key)} — {area_ha:,.0f} ha")

## 2. Forest mask per zone (AlphaEarth clustering + Hansen tree cover)

A raw bounding box mixes forest with roads, water, clearings, and secondary cover, biasing the carbon average. This runs unsupervised k-means on the AlphaEarth embedding for each zone, then labels whichever cluster has the highest mean Hansen tree-cover (2000) as 'forest' and keeps only those pixels for everything below — both the carbon calculation and the embedding signature used for the similarity heatmap.

In [ ]:
CLUSTER_YEAR = 2023
N_CLUSTERS = 4

forest_masks = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    mask, treecover_by_cluster, forest_id = cluster_and_identify_forest(
        CLUSTER_YEAR, geom, n_clusters=N_CLUSTERS
    )
    forest_masks[key] = mask
    print(f"{key}: cluster {forest_id} selected as forest "
          f"(mean tree cover by cluster: {treecover_by_cluster})")

## 3. Aboveground carbon — ESA CCI Biomass (2022), forest-weighted

Continuous, gap-free 100 m maps. Citation: Santoro & Cartus (2025), ESA CCI Biomass v6.0. Each 10 m forest pixel contributes the carbon value of the coarser cell it falls in, weighting proportionally by forest coverage rather than a hard threshold.

In [ ]:
YEAR = 2022  # most recent year available in ESA CCI Biomass v6.0

esa_results = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    esa_raw = get_esa_cci_carbon(YEAR, geom)
    mean_carbon = forest_weighted_mean_carbon(esa_raw, forest_masks[key], geom, scale=10)
    esa_results[key] = mean_carbon
    print(f"{key}: {mean_carbon:,.1f} Mg C/ha (ESA CCI Biomass, forest-weighted)")

## 4. Result 1: aboveground carbon across biomes

The headline chart for the LinkedIn post.

In [ ]:
comparison_df = pd.DataFrame({
    "zone": [get_zone_label(k) for k in ZONES],
    "ESA CCI Biomass": [esa_results[k] for k in ZONES],
})
comparison_df.to_csv("../figures/carbon_comparison.csv", index=False)
comparison_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["darkgreen", "orangered", "steelblue", "goldenrod"]

ax.bar(comparison_df["zone"], comparison_df["ESA CCI Biomass"], color=colors)
ax.set_ylabel("Aboveground carbon (Mg C/ha)")
ax.set_title("Aboveground carbon across biomes — ESA CCI Biomass, forest-weighted")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("../figures/carbon_comparison.png", dpi=200)
plt.show()

## 5. AlphaEarth zone signatures (forest pixels only)

A complementary structural check: how distinct are these four sites in AlphaEarth's 64-dimensional embedding space, once restricted to the same forest mask used for carbon? An earlier version of this used the raw bounding-box mean, which was dominated by regional climate/geography signal (e.g. two nearby Chilean zones looked nearly identical) rather than forest structure — masking to forest-only pixels isolates the structural signal instead.

In [ ]:
EMBEDDING_YEAR = 2023

embedding_img_by_zone = {}
embeddings = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    full_embedding = (
        ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
        .filterDate(f"{EMBEDDING_YEAR}-01-01", f"{EMBEDDING_YEAR + 1}-01-01")
        .filterBounds(geom)
        .mosaic()
        .updateMask(forest_masks[key])
    )
    band_names = full_embedding.bandNames().getInfo()
    stats = full_embedding.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=geom, scale=10, maxPixels=1e13, bestEffort=True,
    ).getInfo()
    embeddings[key] = np.array([stats.get(b, 0.0) or 0.0 for b in band_names])
    print(f"{key}: forest-only embedding vector computed ({len(embeddings[key])} dims)")

## 6. Result 2: zone similarity heatmap for LinkedIn

In [ ]:
labels, sim_matrix = cosine_similarity_matrix(embeddings)
display_labels = [get_zone_label(k) for k in labels]

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(sim_matrix, cmap="YlGnBu", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(display_labels, rotation=30, ha="right")
ax.set_yticklabels(display_labels)
ax.set_title("AlphaEarth embedding similarity between zones (forest pixels only)")

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center",
                 color="white" if sim_matrix[i, j] > 0.5 else "black")

plt.colorbar(im, label="Cosine similarity")
plt.tight_layout()
plt.savefig("../figures/zone_similarity_heatmap.png", dpi=200)
plt.show()

## 7. Combined result for LinkedIn

Headline numbers for the post caption.

In [ ]:
for key in ZONES:
    print(f"{get_zone_label(key)}: {esa_results[key]:,.1f} Mg C/ha (ESA CCI Biomass)")

## 8. Total ecosystem carbon (forest zones)

AGB alone excludes belowground biomass, deadwood/litter, and soil organic carbon (SOC). This adds: BGB via IPCC root:shoot ratios (by biome type), deadwood/litter via an IPCC fraction of AGB, and SOC (0-30 cm) from SoilGrids 250m — see `src/carbon_sources.py` and `src/soil_carbon.py` for the exact factors and their sources.

In [ ]:
from zones import BIOME_TYPE
from carbon_sources import total_forest_carbon
from soil_carbon import list_ocs_bands, mean_soil_carbon

# Diagnostic: confirm the SoilGrids OCS band name before trusting it
print(list_ocs_bands())

In [ ]:
total_carbon_results = {}
for key in ZONES:
    geom = get_zone_geometry(key)
    soc = mean_soil_carbon(geom, scale=250)
    breakdown = total_forest_carbon(esa_results[key], BIOME_TYPE[key], soc)
    total_carbon_results[key] = breakdown
    print(f"{key}: {breakdown}")

## 9. Result 3: total carbon breakdown by pool

In [ ]:
pool_df = pd.DataFrame({
    "zone": [get_zone_label(k) for k in ZONES],
    "AGB": [total_carbon_results[k]["AGB"] for k in ZONES],
    "BGB": [total_carbon_results[k]["BGB"] for k in ZONES],
    "Deadwood/litter": [total_carbon_results[k]["deadwood_litter"] for k in ZONES],
    "SOC (0-30cm)": [total_carbon_results[k]["SOC_0_30cm"] for k in ZONES],
})
pool_df.to_csv("../figures/total_carbon_by_pool.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 6))
pool_df.set_index("zone")[["AGB", "BGB", "Deadwood/litter", "SOC (0-30cm)"]].plot(
    kind="bar", stacked=True, ax=ax,
    color=["forestgreen", "peru", "tan", "saddlebrown"]
)
ax.set_ylabel("Carbon (Mg C/ha)")
ax.set_title("Total ecosystem carbon by pool — forest zones")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("../figures/total_carbon_by_pool.png", dpi=200)
plt.show()

## 10. Wetlands: soil-carbon-dominated comparison

Wetlands store most of their carbon belowground (peat/organic soil), not in aboveground biomass — so this compares them primarily on SOC, alongside the four forest zones for scale. No forest mask applies here (these are not forest ecosystems); the bounding boxes are illustrative placeholders (see `src/zones.py`, `WETLAND_ZONES`).

Limitation: SoilGrids' 0-30 cm depth substantially underestimates true carbon in peat-forming wetlands, where organic layers extend far deeper — treat these numbers as a floor, not a full accounting.

In [ ]:
from zones import WETLAND_ZONES, get_wetland_geometry, get_wetland_label

wetland_soc = {}
for key in WETLAND_ZONES:
    geom = get_wetland_geometry(key)
    wetland_soc[key] = mean_soil_carbon(geom, scale=250)
    print(f"{key}: {wetland_soc[key]:,.1f} Mg C/ha (SOC, 0-30cm)")

## 11. Result 4: forests vs wetlands — soil carbon for LinkedIn

In [ ]:
soc_compare_df = pd.DataFrame({
    "zone": [get_zone_label(k) for k in ZONES] + [get_wetland_label(k) for k in WETLAND_ZONES],
    "soc_Mg_ha": [total_carbon_results[k]["SOC_0_30cm"] for k in ZONES] + [wetland_soc[k] for k in WETLAND_ZONES],
    "type": ["forest"] * len(ZONES) + ["wetland"] * len(WETLAND_ZONES),
})
soc_compare_df.to_csv("../figures/soc_forests_vs_wetlands.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 6))
colors = ["peru" if t == "forest" else "teal" for t in soc_compare_df["type"]]
ax.bar(soc_compare_df["zone"], soc_compare_df["soc_Mg_ha"], color=colors)
ax.set_ylabel("Soil organic carbon, 0-30 cm (Mg C/ha)")
ax.set_title("Soil carbon: forests vs wetlands (floor estimate for wetlands)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("../figures/soc_forests_vs_wetlands.png", dpi=200)
plt.show()

## 12. Where is the carbon? Dominant pool by ecosystem

The core message: forests store most of their carbon aboveground (in living biomass); wetlands store almost all of theirs belowground (in soil/peat). This makes that explicit by showing each ecosystem's carbon as a percentage split between 'aboveground + deadwood' and 'belowground (soil)'.

In [ ]:
pool_share_records = []
for key in ZONES:
    b = total_carbon_results[key]
    aboveground = b["AGB"] + b["BGB"] + b["deadwood_litter"]
    belowground = b["SOC_0_30cm"]
    pool_share_records.append({
        "zone": get_zone_label(key), "ecosystem": "forest",
        "aboveground_pct": 100 * aboveground / b["total"],
        "belowground_pct": 100 * belowground / b["total"],
    })

for key in WETLAND_ZONES:
    # Wetlands: no AGB estimate here (not a forest biomass product target);
    # treat as ~100% belowground for this comparison, per the wetland's
    # own ecology (see README limitation on peat depth).
    pool_share_records.append({
        "zone": get_wetland_label(key), "ecosystem": "wetland",
        "aboveground_pct": 0.0,
        "belowground_pct": 100.0,
    })

pool_share_df = pd.DataFrame(pool_share_records)
pool_share_df.to_csv("../figures/carbon_pool_share.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 6))
pool_share_df.set_index("zone")[["aboveground_pct", "belowground_pct"]].plot(
    kind="barh", stacked=True, ax=ax, color=["forestgreen", "saddlebrown"]
)
ax.set_xlabel("Share of total carbon (%)")
ax.set_title("Where is the carbon stored? Aboveground vs belowground")
ax.legend(["Aboveground + deadwood", "Belowground (soil)"], loc="lower right")
plt.tight_layout()
plt.savefig("../figures/carbon_pool_share.png", dpi=200)
plt.show()

## 13. What's at risk: total carbon (not just density) per zone

Mg C/ha tells you concentration, not what's actually at stake in a given place. This multiplies each zone's carbon density by its real area to get total tonnes, converted to CO2-equivalent (x 3.667, the molecular weight ratio of CO2 to C). This is the 'stock at risk' framing used in wetland/forest conservation messaging: the amount that could be released if the ecosystem is destroyed or drained — not a claim that 100% of it would be emitted instantly, since decomposition and emission dynamics vary by disturbance type.

In [ ]:
CO2_PER_C = 3.667  # molecular weight ratio, CO2 (44) / C (12)

risk_records = []
for key in ZONES:
    geom = get_zone_geometry(key)
    area_ha = geom.area().divide(10000).getInfo()
    total_c_mg_ha = total_carbon_results[key]["total"]
    total_c_tonnes = total_c_mg_ha * area_ha
    risk_records.append({
        "zone": get_zone_label(key), "ecosystem": "forest",
        "area_ha": area_ha, "total_C_Mg": total_c_tonnes,
        "total_CO2e_Mg": total_c_tonnes * CO2_PER_C,
    })

for key in WETLAND_ZONES:
    geom = get_wetland_geometry(key)
    area_ha = geom.area().divide(10000).getInfo()
    total_c_tonnes = wetland_soc[key] * area_ha
    risk_records.append({
        "zone": get_wetland_label(key), "ecosystem": "wetland",
        "area_ha": area_ha, "total_C_Mg": total_c_tonnes,
        "total_CO2e_Mg": total_c_tonnes * CO2_PER_C,
    })

risk_df = pd.DataFrame(risk_records)
risk_df.to_csv("../figures/carbon_at_risk.csv", index=False)
risk_df

## 14. Result 5: carbon at risk for LinkedIn

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
colors = ["peru" if e == "forest" else "teal" for e in risk_df["ecosystem"]]
ax.bar(risk_df["zone"], risk_df["total_CO2e_Mg"], color=colors)
ax.set_ylabel("Total CO2-equivalent at risk (Mg)")
ax.set_title("Carbon at risk if the ecosystem is lost")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("../figures/carbon_at_risk.png", dpi=200)
plt.show()

print("\nHeadline numbers:")
for _, row in risk_df.iterrows():
    print(f"  {row['zone']}: {row['total_CO2e_Mg']:,.0f} Mg CO2e at risk over {row['area_ha']:,.0f} ha")

## 15. Is the radiata plantation stand young or old?

Two independent signals, cross-checked: (1) years since the last Hansen-detected stand-replacement disturbance (a pixel with no recorded loss since 2000 is age-censored, not necessarily old), and (2) canopy height (ETH 2020) converted to an approximate age via an illustrative Pinus radiata height-age curve for Chile. This matters for interpreting the carbon numbers: a young rotation and a mature stand of the 'same' plantation can have very different AGB.

In [ ]:
from stand_age_utils import get_years_since_loss, get_canopy_height, height_to_age

radiata_geom = get_zone_geometry("radiata_biobio")
radiata_forest_mask = forest_masks["radiata_biobio"]

years_since_loss = get_years_since_loss(radiata_geom).updateMask(radiata_forest_mask)
canopy_height = get_canopy_height(radiata_geom).updateMask(radiata_forest_mask)

In [ ]:
pct_with_recorded_loss = years_since_loss.mask().reduceRegion(
    reducer=ee.Reducer.mean(), geometry=radiata_geom, scale=30, maxPixels=1e13, bestEffort=True,
).get("years_since_loss").getInfo()

mean_years_since_loss = years_since_loss.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=radiata_geom, scale=30, maxPixels=1e13, bestEffort=True,
).get("years_since_loss").getInfo()

mean_canopy_height = canopy_height.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=radiata_geom, scale=10, maxPixels=1e13, bestEffort=True,
).get("canopy_height_m").getInfo()

height_derived_age = height_to_age(mean_canopy_height) if mean_canopy_height else None

print(f"Forest pixels with a recorded Hansen loss event since 2000: {pct_with_recorded_loss:.0%}")
print(f"Mean years since last recorded loss (where dated): {mean_years_since_loss:.1f} years")
print(f"Mean canopy height (ETH 2020): {mean_canopy_height:.1f} m")
print(f"Height-derived approximate age: {height_derived_age:.1f} years")

## 16. Result 6: stand-age mosaic for LinkedIn

A histogram of years-since-loss shows the rotation mosaic typical of industrial plantation management, rather than a single uniform age.

In [ ]:
hist = years_since_loss.reduceRegion(
    reducer=ee.Reducer.fixedHistogram(0, 25, 25),
    geometry=radiata_geom, scale=30, maxPixels=1e13, bestEffort=True,
).get("years_since_loss").getInfo()

bins = [b[0] for b in hist]
counts = [b[1] for b in hist]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(bins, counts, width=0.9, color="goldenrod", align="edge")
ax.set_xlabel("Years since last recorded stand-replacement disturbance")
ax.set_ylabel("Pixel count (30 m)")
ax.set_title("Radiata plantation — rotation-age mosaic (Biobio)")
plt.tight_layout()
plt.savefig("../figures/radiata_stand_age_histogram.png", dpi=200)
plt.show()